# Phase 3b: Feature diagnostics — why Ridge exploded, and what it reveals

Notebook 3 produced one anomaly worth chasing to the bottom: Ridge posted
RMSE of 81 → 258 → 574 ft while the tree models sat at 7–14 ft. That is not
"Ridge is a weak learner" — a 574 ft error on a target that moves ~0.3 ft/row
is a sign something is structurally wrong with the **feature matrix**, and that
something affects every model, not just Ridge.

This notebook investigates properly. The conclusion turns out to be bigger than
the Ridge bug: **the absolute geometric features (`MD, X, Y, Z`) cannot
generalize across wells**, and the most predictive-looking signal (ΔTVT
autocorrelation) is **illusory for forward extrapolation**. Both findings point
to the same place — the lateral's own features can't carry the forward tail; the
typewell reference (notebook 4) must.

We build each claim from evidence, not assertion.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")


def tail_mask(n, frac):
    k = round(n * frac)
    m = np.zeros(n, bool)
    if k:
        m[n - k :] = True
    return m


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def reconstruct(g, mask, dpred, target="TVT"):
    tvt = g[target].values.astype(float)
    out = tvt.copy()
    last = np.where(~mask)[0][-1]
    run = tvt[last]
    for i in np.where(mask)[0]:
        run += dpred[i]
        out[i] = run
    return out


# Load a handful of real wells for diagnosis (full set not needed to see structure).
files = sorted((CLEAN_DIR / "train").glob("*__horizontal_well.csv"))[:30]
wells = [pd.read_csv(f, dtype={"well_id": str}) for f in files]
print(f"loaded {len(wells)} wells for diagnosis")
df0 = wells[0].sort_values("MD").reset_index(drop=True)
print("example well:", df0["well_id"].iloc[0], "| rows:", len(df0))

## Finding 1 — The feature matrix is catastrophically ill-conditioned

The notebook-3 feature set was `[GR_z, gr_missing, gr_flatline, traj_teleport,
MD, X, Y, Z]`. Two immediate problems, both visible in one table:
- the continuous features span **six orders of magnitude** (X ~ 3e6, GR_z ~ 1),
- some flags are **constant within a well** (no flatline/teleport rows), so they
  add zero-variance columns.

A linear model must find coefficients ~1e-7 to map million-scale inputs to a
0.3-scale target; in that regime, ordinary conditioning problems blow up.

In [ ]:
FEATS = ["GR_z", "gr_missing", "gr_flatline", "traj_teleport", "MD", "X", "Y", "Z"]
stat = pd.DataFrame(
    {
        "mean": df0[FEATS].astype(float).mean(),
        "std": df0[FEATS].astype(float).std(),
        "var": df0[FEATS].astype(float).var(),
    }
)
stat["constant?"] = stat["var"] == 0
print(stat)
print(
    "\nfeature scale span (max std / min nonzero std):",
    f"{stat['std'].max() / stat.loc[stat['std']>0, 'std'].min():.2e}",
)

## Finding 2 — The real culprit is collinearity, not just scale

Standardizing would fix scale. It does **not** fix the deeper problem: in a
near-horizontal lateral, **MD and the along-hole coordinate are essentially the
same axis**. The correlation below is ~0.9997. Two near-duplicate columns make
the linear system singular; coefficients become a huge-magnitude tug-of-war that
nearly cancels, and the residual error integrates into the hundreds of feet we
saw. (Trees are immune — they split, they don't invert a matrix.)

In [ ]:
geom = ["MD", "X", "Y", "Z"]
print("geometry correlation matrix (one well):")
print(df0[geom].astype(float).corr().round(4))

# centered singular values -> honest conditioning of the continuous block
Xc = df0[geom].astype(float).values
Xc = Xc - Xc.mean(0)
s = np.linalg.svd(Xc, compute_uv=False)
print("\nsingular values:", np.round(s, 3))
print("condition (sv_max/sv_min):", f"{s[0]/s[-1]:.2e}")
print("=> a near-zero singular value means one geometry axis is redundant")

In [ ]:
# Demonstrate the explosion AND that standardizing alone does not save it.
def fit_eval_ridge(df, frac, feats, standardize):
    df = df.sort_values("MD").reset_index(drop=True)
    m = tail_mask(len(df), frac)
    delta = df["TVT"].diff().fillna(0.0).values
    X = df[feats].astype(float).values
    Xtr, ytr, Xev = X[~m], delta[~m], X[m]
    if standardize:
        sc = StandardScaler().fit(Xtr)
        Xtr, Xev = sc.transform(Xtr), sc.transform(Xev)
    r = Ridge(alpha=1.0).fit(Xtr, ytr)
    dp = np.zeros(len(df))
    dp[m] = r.predict(Xev)
    tp = reconstruct(df, m, dp)
    bias = dp[m].mean() - delta[m].mean()
    return rmse(tp[m], df["TVT"].values[m]), bias, m.sum()


for label, std in [("raw", False), ("standardized", True)]:
    r, bias, n = fit_eval_ridge(df0, 0.25, FEATS, std)
    print(
        f"Ridge {label:13s}: RMSE {r:8.2f} ft | per-row ΔTVT bias {bias:+.5f} "
        f"-> {bias*n:+.1f} ft cumulative drift over {n} rows"
    )
print("\n=> standardizing rescales but leaves the MD/Y redundancy; RMSE stays large.")

## Finding 3 — Absolute geometry cannot generalize across wells (the real flaw)

This is the finding that matters most, and it indicts the **tree models too**,
not just Ridge. `X` and `Y` are **geographic coordinates** — this well sits near
X≈3.0e6, Y≈1.07e6. A model that learns "ΔTVT behaves thus at Y=1,071,418" has
learned a fact about *this patch of ground*, not about wellbore physics. A test
well lives at different coordinates, so the learned mapping is unusable there.

And `MD` is, by the EDA's uniform-step finding, just a **row counter** (ΔMD ≡ 1).
Absolute MD ≈ 14,000 transfers nothing either.

In [ ]:
print("Per-well coordinate centers (first 8 wells) — note how each sits elsewhere:")
for w in wells[:8]:
    g = w.sort_values("MD")
    print(
        f"  {g['well_id'].iloc[0]}: X~{g['X'].mean():,.0f}  Y~{g['Y'].mean():,.0f}  "
        f"Z~{g['Z'].mean():,.0f}  MD {g['MD'].min():.0f}-{g['MD'].max():.0f}"
    )

dmd = df0["MD"].diff().dropna()
print(f"\nΔMD within a well: min={dmd.min()}, max={dmd.max()} -> MD is a row index, not a feature")
print("\n=> absolute MD/X/Y/Z are per-well constants-in-disguise; the model cannot")
print("   transfer them to a test well at different coordinates.")

### What *would* transfer: local, relative quantities

The signal that could generalize is **local geometry** — how the path is
*changing*, not where it *is* — and lithology via `GR_z`. Below, `dZ/dMD`
(inclination proxy) correlates strongly with ΔTVT. So do lagged ΔTVT values.
But the lag is a trap — see Finding 4.

In [ ]:
g = df0.copy()
g["dZ_dMD"] = g["Z"].diff() / g["MD"].diff()
g["dTVT"] = g["TVT"].diff()
g["dTVT_lag1"] = g["dTVT"].shift(1)
print("Correlation with ΔTVT (the actual per-row target):")
for c in ["GR_z", "dZ_dMD", "dTVT_lag1"]:
    cc = pd.concat([g[c], g["dTVT"]], axis=1).dropna().corr().iloc[0, 1]
    print(f"  {c:12s} {cc:+.3f}")

## Finding 4 — The ΔTVT autocorrelation is illusory for forward prediction

`dTVT_lag1` correlates ~0.99 with ΔTVT — tantalizing, but a trap. In the eval
tail you do **not** observe the previous true ΔTVT; you only have your own
predictions, so using the lag means persisting the last known slope forward. The
test below shows persisting the last (or smoothed) ΔTVT is **far worse** than
holding flat, because the lateral *flattens* in the eval zone (true eval ΔTVT
mean ≈ 0) while the last observed slope is slightly nonzero — and that small bias
integrates into a large drift.

In [ ]:
def persist_models(df, frac, win=50):
    df = df.sort_values("MD").reset_index(drop=True)
    m = tail_mask(len(df), frac)
    tvt = df["TVT"].values.astype(float)
    dtvt = df["TVT"].diff().fillna(0).values
    last = np.where(~m)[0][-1]
    res = {}
    # flat hold (floor)
    p = tvt.copy()
    p[m] = tvt[last]
    res["floor_flat"] = rmse(p[m], tvt[m])
    # persist last ΔTVT
    p = tvt.copy()
    run = tvt[last]
    for i in np.where(m)[0]:
        run += dtvt[last]
        p[i] = run
    res["persist_last_delta"] = rmse(p[m], tvt[m])
    # persist smoothed ΔTVT
    sd = dtvt[max(0, last - win + 1) : last + 1].mean()
    p = tvt.copy()
    run = tvt[last]
    for i in np.where(m)[0]:
        run += sd
        p[i] = run
    res["persist_smoothed_delta"] = rmse(p[m], tvt[m])
    return res, dtvt[m].mean(), dtvt[last]


agg = {}
for w in wells:
    r, true_mean, last_d = persist_models(w, 0.25)
    for k, v in r.items():
        agg.setdefault(k, []).append(v)
print("Mean RMSE across", len(wells), "wells (mask 25%):")
for k, v in agg.items():
    print(f"  {k:24s} {np.mean(v):7.3f} ft")
print("\n=> persisting any slope loses to holding flat: the lateral's own trend does")
print("   NOT extrapolate. The forward signal is not in the lateral's past.")

## Conclusions

**On the Ridge bug specifically:**
1. The feature matrix mixed 1e6-scale coordinates with unit-scale signals and
   included zero-variance flag columns — terrible conditioning.
2. The deeper cause is **MD/coordinate collinearity (~0.9997)**; standardizing
   does not fix it. Ridge's coefficients blow up and the per-row bias integrates
   into hundreds of feet.

**The bigger, model-agnostic findings:**
3. **Absolute `MD, X, Y, Z` cannot generalize across wells** — X/Y are geographic
   positions, MD is a row counter. Every notebook-3 model was partly fed
   untransferable features; trees survived only by ignoring them.
4. **ΔTVT autocorrelation is useless for the forward tail** — you can't observe
   the lagged truth at inference, and persisting the last slope loses to holding
   flat because the lateral flattens.

**What this means for the modelling, concretely:**
- **Drop absolute coordinates.** Replace with local/relative features:
  `dZ/dMD` (inclination), `dX/dMD`, curvature, GR-window statistics — quantities
  defined by *change*, which transfer across wells.
- **Stop expecting the lateral's own history to carry the tail.** It doesn't.
  This is the third independent confirmation that the **typewell match
  (notebook 4)** is the load-bearing signal — it re-anchors prediction against an
  absolute geological reference instead of integrating a drifting delta.
- For any future linear baseline, standardize **and** drop one of each collinear
  pair; but trees on relative features are the more promising baseline.

The floor to beat remains `floor_hold_last` (≈ 4.9 / 8.1 / 9.9 ft at 10/25/40%
mask on the full set). Notebook 4 should target it with typewell matching, using
**relative** geometric features only.